# Legal Outcome Prediction Pipeline — AWS SageMaker Studio

**Before running:**
1. Launch this notebook on a GPU instance: `ml.g4dn.xlarge` (T4, 16 GB) or `ml.g5.xlarge` (A10G, 24 GB)
2. Upload the `NLPPW` project folder via the JupyterLab file browser **or** git clone it into `/home/sagemaker-user/`
3. Upload `echr-args-dataset.zip` to your S3 bucket
4. Set your HuggingFace token in the cell below

Run all cells in order.

In [1]:
# ── Configure these before running ──────────────────────────────────────────
S3_BUCKET      = ''          # <-- your S3 bucket
S3_DATASET_KEY = ''        # path inside the bucket
HF_TOKEN       = ''       # <-- your HF token
# ────────────────────────────────────────────────────────────────────────────

import os
import sys
from pathlib import Path

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow warnings
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'  # Disable oneDNN custom operations

import warnings
warnings.filterwarnings('ignore')

# Prevent transformers from loading TensorFlow (pre-installed in SageMaker base image)
os.environ['USE_TF'] = '0'

PROJECT_PATH = Path('/home/sagemaker-user/NLPPW')  # adjust if placed elsewhere
os.chdir(PROJECT_PATH)
sys.path.insert(0, str(PROJECT_PATH))

os.environ['HF_TOKEN'] = HF_TOKEN

print('Working directory:', os.getcwd())

Working directory: /home/sagemaker-user/NLPPW


In [2]:
%%capture
!pip install \
    'transformers>=4.40.0' \
    'sentence-transformers>=2.7.0' \
    'datasets>=2.19.0' \
    'huggingface-hub' \
    'scikit-learn>=1.4.0' \
    'interpret>=0.6.0' \
    'numpy>=1.26.0' \
    'pandas>=2.2.0' \
    'matplotlib>=3.8.0' \
    'seaborn>=0.13.0' \
    'tqdm>=4.66.0' \
    'joblib>=1.4.0'
print('Dependencies installed.')

In [3]:
import boto3
import zipfile
import config

ZIP_LOCAL  = PROJECT_PATH / 'echr-args-dataset.zip'
EXTRACT_TO = Path('/home/sagemaker-user/NLPPW')

# Download zip from S3 if not already present
if not ZIP_LOCAL.exists():
    print(f'Downloading s3://{S3_BUCKET}/{S3_DATASET_KEY} ...')
    boto3.client('s3').download_file(S3_BUCKET, S3_DATASET_KEY, str(ZIP_LOCAL))
    print('Download complete.')
else:
    print(f'Zip already present at {ZIP_LOCAL} — skipping download.')

# Extract
if not EXTRACT_TO.exists():
    print(f'Extracting to {EXTRACT_TO} ...')
    with zipfile.ZipFile(ZIP_LOCAL, 'r') as z:
        z.extractall(EXTRACT_TO)
    print('Extraction complete.')
else:
    print(f'Already extracted at {EXTRACT_TO} — skipping.')


config.NEW_DATASET_DIR = EXTRACT_TO / 'all-data'
n_files = len(list(config.NEW_DATASET_DIR.glob('*.json')))
print(f'Dataset ready: {n_files} JSON files')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Zip already present at /home/sagemaker-user/NLPPW/echr-args-dataset.zip — skipping download.
Already extracted at /home/sagemaker-user/NLPPW — skipping.
Dataset ready: 12947 JSON files


In [4]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
NVIDIA A10G
VRAM: 23.7 GB


In [5]:
print(f'Dataset dir  : {config.NEW_DATASET_DIR}')
print(f'Stage 1 model: {config.LEGALBERT_MODEL}')
print(f'Stage 2 model: {config.SENTENCE_TRANSFORMER_MODEL}')
print(f'Classifier   : {config.CLASSIFIER_TYPE}')
print(f'Epochs       : {config.S1_EPOCHS}')
print(f'Batch size   : {config.S1_BATCH_SIZE}')
print(f'Fact negatives: {config.FACT_NEGATIVES}  (sim threshold: {config.FACT_SIM_THRESHOLD})')
print(f'Dynamic top-k : {config.DYNAMIC_TOPK}  (alpha: {config.DYNAMIC_TOPK_ALPHA}, floor: {config.PREMISE_FLOOR})')

Dataset dir  : /home/sagemaker-user/NLPPW/all-data
Stage 1 model: /home/sagemaker-user/NLPPW/outputs/stage1_legalbert/checkpoint-best
Stage 2 model: sentence-transformers/all-mpnet-base-v2
Classifier   : svm
Epochs       : 3
Batch size   : 32
Fact negatives: True  (sim threshold: 0.6)
Dynamic top-k : True  (alpha: 0.5, floor: 0.5)


## Stage 1 — Fine-tune LegalBERT (run once)
Fine-tunes LegalBERT on the 12947 case ECtHR dataset.
Splitting is done at the **case_id level** so no case leaks across train/val/test.

**Similarity-filtered fact negatives** (`config.FACT_NEGATIVES`): before fine-tuning, fact sentences
that are dissimilar to any party argument (cosine sim < `FACT_SIM_THRESHOLD`) are added as NON_PREMISE.
Run the inspection cell below to review what gets kept vs filtered.

The checkpoint is saved to `outputs/stage1_legalbert/checkpoint-best`.
**Skip this cell on subsequent runs** — `config.py` auto-detects the saved checkpoint.

In [6]:
from stage1_argument_mining.fact_filter import inspect_fact_negatives

inspect_fact_negatives(n_cases=5, n_sentences=5)


Case: 001-119967  (18 premises, 48 facts)
  KEPT as NON_PREMISE (46):
    sim=0.414  I.  THE CIRCUMSTANCES OF THE CASE
    sim=0.498  The facts of the case, as submitted by the parties, may be summarised as follows.
    sim=0.442  A.  Civil and enforcement proceedings
    sim=0.386  All applicants were former employees of “LETEKS” u stečaju (the debtor), which was, at the relevant time, a company predominantly comprised of socially-owned capital.
    sim=0.404  On 18 April 2008 the Municipal Court in Leskovac ordered the debtor to pay them:
  FILTERED OUT (2):
    sim=0.627  Since the debtor failed to fulfil its contractual obligations towards employees, on unspecified date, the applicants instituted civil proceedings against it.
    sim=0.627  Since the debtor failed to fulfil its contractual obligations towards employees, on unspecified date, the applicants instituted civil proceedings against it.

Case: 001-66687  (36 premises, 108 facts)
  KEPT as NON_PREMISE (46):
    sim=0.486  

In [6]:
import importlib
from pathlib import Path

checkpoint = config.OUTPUT_DIR / 'stage1_legalbert' / 'checkpoint-best'

if checkpoint.exists():
    print(f'Fine-tuned checkpoint found at:\n  {checkpoint}')
    print('Skipping fine-tuning — delete that folder to retrain.')
else:
    print('No checkpoint found. Starting fine-tuning...')
    from stage1_argument_mining.finetune_legalbert import finetune
    finetune()
    importlib.reload(config)
    print(f'\nLEGALBERT_MODEL is now: {config.LEGALBERT_MODEL}')

Fine-tuned checkpoint found at:
  /home/sagemaker-user/NLPPW/outputs/stage1_legalbert/checkpoint-best
Skipping fine-tuning — delete that folder to retrain.


In [11]:
# Evaluate the Stage 1 checkpoint on test set
import numpy as np
from sklearn.metrics import classification_report, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from stage1_argument_mining.finetune_legalbert import prepare_data, ID2LABEL, LABEL2ID

checkpoint = config.OUTPUT_DIR / 'stage1_legalbert' / 'checkpoint-best'

if checkpoint.exists():
    print(f'Evaluating checkpoint: {checkpoint}')
    
    # Load test data
    _, _, test_rows = prepare_data()
    
    # Tokenize
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    test_ds = Dataset.from_dict({
        "text": [r["text"] for r in test_rows],
        "label": [r["label"] for r in test_rows],
    })
    
    def tokenize_batch(batch):
        return tokenizer(batch["text"], truncation=True, max_length=config.S1_MAX_SEQ_LEN, padding="max_length")
    
    test_ds = test_ds.map(tokenize_batch, batched=True, desc="Tokenizing test set")
    
    # Load model
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID)
    
    # Evaluate
    trainer = Trainer(
        model=model,
        args=TrainingArguments(output_dir=str(config.OUTPUT_DIR / "temp"), per_device_eval_batch_size=config.S1_BATCH_SIZE, seed=config.RANDOM_SEED)
    )
    
    out = trainer.predict(test_ds)
    preds = np.argmax(out.predictions, axis=-1)
    
    print("\n" + "="*60)
    print("Stage 1 LegalBERT - Test Set Results")
    print("="*60)
    print(classification_report(out.label_ids, preds, target_names=["NON_PREMISE", "PREMISE"]))
    
    f1_macro = f1_score(out.label_ids, preds, average="macro", zero_division=0)
    f1_binary = f1_score(out.label_ids, preds, average="binary", zero_division=0)
    print(f"\nMacro F1:  {f1_macro:.4f}")
    print(f"Binary F1: {f1_binary:.4f}")
    print("="*60)
else:
    print("No checkpoint found. Run fine-tuning first.")

Evaluating checkpoint: /home/sagemaker-user/NLPPW/outputs/stage1_legalbert/checkpoint-best
Loading new dataset from /home/sagemaker-user/NLPPW/all-data ...
  8364 unique case_ids loaded (argument units)
  Loading cached fact negatives from /home/sagemaker-user/NLPPW/outputs/fact_negatives_filtered.json
  + 457905 similarity-filtered fact negatives from 8315 cases
  Case-level split: 5856 train / 1254 val / 1254 test cases

Split sizes (train after balancing):
  train :  412540 sentences  (206270 premises / 206270 non-premises)
  val   :  121109 sentences  (44805 premises / 76304 non-premises)
  test  :  122092 sentences  (44595 premises / 77497 non-premises)


Tokenizing test set:   0%|          | 0/122092 [00:00<?, ? examples/s]

2026-05-01 20:00:36.024259: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777665636.047721     758 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777665636.058233     758 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777665636.232572     758 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777665636.232600     758 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777665636.232614     758 computation_placer.cc:177] computation placer alr


Stage 1 LegalBERT - Test Set Results
              precision    recall  f1-score   support

 NON_PREMISE       0.99      0.95      0.97     77497
     PREMISE       0.93      0.98      0.95     44595

    accuracy                           0.96    122092
   macro avg       0.96      0.97      0.96    122092
weighted avg       0.96      0.96      0.96    122092


Macro F1:  0.9603
Binary F1: 0.9504


## Stage 1 — Argument Mining
Loads the fine-tuned LegalBERT and classifies each sentence as a premise or not.

In [7]:
from pathlib import Path
from data.data_loader import get_dataset
from stage1_argument_mining.sequence_filter import run_stage1, load_stage1, print_stage1_stats

dataset = get_dataset()

if Path(config.STAGE1_CACHE).exists():
    print(f'Cached premises found at {config.STAGE1_CACHE} — skipping extraction.')
    print('Delete this file to re-run Stage 1 extraction.')
    stage1_output = load_stage1()
else:
    from stage1_argument_mining.argument_extractor import LegalBERTArgumentExtractor
    extractor = LegalBERTArgumentExtractor()
    stage1_output = run_stage1(dataset, extractor)

print_stage1_stats(stage1_output)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'lex_glue' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Preprocessing dataset:   0%|          | 0/9000 [00:00<?, ? examples/s]

Preprocessing dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Preprocessing dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Cached premises found at /home/sagemaker-user/NLPPW/outputs/stage1_extracted_premises.json — skipping extraction.
Delete this file to re-run Stage 1 extraction.

Stage 1 – Argument Mining Statistics
  [train] cases= 9000  total_premises=102067  avg= 11.3  zero_premise_cases=811
  [  val] cases= 1000  total_premises= 13664  avg= 13.7  zero_premise_cases=49
  [ test] cases= 1000  total_premises= 14284  avg= 14.3  zero_premise_cases=39


In [8]:
from stage1_argument_mining.argument_extractor import _split_sentences, LegalBERTArgumentExtractor

print('Stage 1 model:', config.LEGALBERT_MODEL)
print(f'Dynamic top-k: {config.DYNAMIC_TOPK}  (alpha={config.DYNAMIC_TOPK_ALPHA}, floor={config.PREMISE_FLOOR})')
print()

# Build extractor only for diagnostics if not already loaded
if 'extractor' not in dir() or extractor is None:
    extractor = LegalBERTArgumentExtractor()

sample_paragraphs = dataset['test'][9]['paragraphs']
print(f'Probing first test case ({len(sample_paragraphs)} paragraphs), raw scores:\n')
for para in sample_paragraphs[:]:
    for sent in _split_sentences(para):
        is_p, conf = extractor.predict_sentence(sent)
        label = 'PREMISE' if is_p else '---'
        print(f'  {conf:.3f}  {label:<10}  {sent[:]}')

premises = extractor.extract_premises(sample_paragraphs)
print(f'\nDynamic top-k selected {len(premises)} premises:')
for p in premises:
    print(f'  {p["confidence"]:.3f}  [para {p["paragraph_id"]}]  {p["sentence"][:120]}')

Stage 1 model: /home/sagemaker-user/NLPPW/outputs/stage1_legalbert/checkpoint-best
Dynamic top-k: True  (alpha=0.5, floor=0.5)

Probing first test case (12 paragraphs), raw scores:

  0.416  ---         The applicant, S.S.
  0.000  ---         Yeniköy Konut Yapı Kooperatifi, is a housing construction cooperative under Turkish law operating in İzmir.
  0.006  ---         In 1993, a third party cooperative bought a plot of land measuring 12,000 square metres and the title deed of the land was registered in its name.
  0.007  ---         In 1995, construction works started on the land in question.
  0.001  ---         In 2000, the forest administration initiated proceedings before the Menderes Civil Court of First Instance for the annulment of the title deed to the land, alleging that it was part of the public forest area.
  0.967  PREMISE     In the meantime, the third party cooperative had merged with the applicant cooperative and the land had been registered in the Land Registry in the

## Stage 2 — Outcome Prediction
Embeds extracted premises with Sentence-BERT and trains the classifier.

In [9]:
from pathlib import Path
from stage2_outcome_prediction.embedder import PremiseEmbedder
from stage2_outcome_prediction.classifier import (
    train_classifier, save_classifier, load_classifier, quick_eval)

embedder = PremiseEmbedder()

X_train, y_train = embedder.prepare_split(stage1_output['train'])
X_val,   y_val   = embedder.prepare_split(stage1_output['val'])
X_test,  y_test  = embedder.prepare_split(stage1_output['test'])

if Path(config.MODEL_CACHE).exists():
    print(f'Cached classifier found at {config.MODEL_CACHE} — skipping training.')
    print('Delete this file to retrain.')
    clf = load_classifier()
else:
    clf = train_classifier(X_train, y_train)
    save_classifier(clf)

quick_eval(clf, X_val, y_val)

Batches:   0%|          | 0/1595 [00:00<?, ?it/s]

Batches:   0%|          | 0/214 [00:00<?, ?it/s]

Batches:   0%|          | 0/224 [00:00<?, ?it/s]

Cached classifier found at /home/sagemaker-user/NLPPW/outputs/stage2_classifier.joblib — skipping training.
Delete this file to retrain.
Val  Macro-F1=0.5963  Micro-F1=0.6357


{'macro_f1': 0.5963407431757558, 'micro_f1': 0.6357142857142857}

## Evaluation

In [10]:
from stage2_outcome_prediction.classifier import predict, predict_with_thresholds, tune_thresholds
from evaluation.metrics import (compare_classifiers, per_article_f1,
                                 print_per_article_f1, train_baseline_classifier)
from evaluation.qualitative_review import run_qualitative_review
from data.data_loader import ARTICLE_NAMES

# Tune per-article thresholds on validation set
print('Tuning per-article thresholds on validation set...')
thresholds = tune_thresholds(clf, X_val, y_val)
for name, t in zip(ARTICLE_NAMES, thresholds):
    print(f'  {name:<12} threshold={t:+.4f}')

# Predict with tuned thresholds
y_pred_pipeline = predict_with_thresholds(clf, X_test, thresholds)

# Baseline (raw text, default thresholds)
y_pred_baseline, _ = train_baseline_classifier(stage1_output['train'], stage1_output['test'], embedder)

comparison    = compare_classifiers(y_test, y_pred_baseline, y_pred_pipeline)
pa_f1_results = per_article_f1(y_test, y_pred_pipeline)
print_per_article_f1(pa_f1_results)

run_qualitative_review(stage1_output['test'], X_test, clf, embedder)

Tuning per-article thresholds on validation set...
  Article 2    threshold=+0.1550
  Article 3    threshold=+0.2552
  Article 5    threshold=+0.0615
  Article 6    threshold=+0.0109
  Article 8    threshold=+0.7555
  Article 9    threshold=-0.0425
  Article 10   threshold=+0.3337
  Article 11   threshold=+0.0176
  Article 14   threshold=+0.0423
  P1-1         threshold=+0.3087


Batches:   0%|          | 0/141 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]


──────────────────────────────────────────────────
  BASELINE (raw text)
──────────────────────────────────────────────────
  macro_f1              : 0.4887
  micro_f1              : 0.5440
  macro_precision       : 0.4138
  macro_recall          : 0.6945
  micro_precision       : 0.4593
  micro_recall          : 0.6669
  hamming_loss          : 0.1264
──────────────────────────────────────────────────

──────────────────────────────────────────────────
  PIPELINE (extracted premises)
──────────────────────────────────────────────────
  macro_f1              : 0.4949
  micro_f1              : 0.5583
  macro_precision       : 0.4822
  macro_recall          : 0.5336
  micro_precision       : 0.5381
  micro_recall          : 0.5800
  hamming_loss          : 0.1037
──────────────────────────────────────────────────

──────────────────────────────────────────────────
  Δ (pipeline − baseline)
──────────────────────────────────────────────────
  macro_f1              : 0.0062
  micro_f1    

[{'verdict': {'case_id': 952,
   'predicted_articles': ['Article 2', 'Article 3', 'Article 5'],
   'true_articles': ['Article 2', 'Article 3', 'Article 5'],
   'justifications': [{'article': 'Article 2',
     'supporting_premises': [{'sentence': 'The servicemen were of Slavic appearance and spoke unaccented Russian; some of them were wearing balaclavas.',
       'paragraph_id': 3,
       'sentence_id': 2,
       'confidence': 0.9982,
       'attribution': 0.05463},
      {'sentence': 'Her complaints were forwarded to the investigators.',
       'paragraph_id': 33,
       'sentence_id': 1,
       'confidence': 0.9991,
       'attribution': 0.04398},
      {'sentence': 'The investigators’ decision stated that her husband Mr Zaurbek Umarov had been abducted together with Mr Mukhtarov.',
       'paragraph_id': 16,
       'sentence_id': 2,
       'confidence': 0.8795,
       'attribution': 0.03899}]},
    {'article': 'Article 3',
     'supporting_premises': [{'sentence': 'She confirmed the 

## Classifier Comparison
Trains SVM, Decision Tree, and EBM on the same premise embeddings and compares against the raw-text baseline.

In [ ]:
from tqdm import tqdm
from stage2_outcome_prediction.classifier import train_classifier, tune_thresholds, predict_with_thresholds
from evaluation.metrics import compute_metrics, per_article_f1, EVAL_ARTICLE_NAMES, train_baseline_classifier

classifiers = {}
results = {}
original_clf_type = config.CLASSIFIER_TYPE

for clf_type in tqdm(['svm', 'decision_tree', 'ebm'], desc='Training classifiers'):
    config.CLASSIFIER_TYPE = clf_type
    c = train_classifier(X_train, y_train)
    classifiers[clf_type] = c
    
    # Tune thresholds on validation set for fair comparison
    thresholds = tune_thresholds(c, X_val, y_val)
    y_pred = predict_with_thresholds(c, X_test, thresholds)
    
    results[clf_type] = compute_metrics(y_test, y_pred)
    results[clf_type]['per_article'] = per_article_f1(y_test, y_pred)
    macro = results[clf_type]['macro_f1']
    micro = results[clf_type]['micro_f1']
    print(f'  {clf_type:<15} macro_f1={macro:.4f}  micro_f1={micro:.4f}')

print('\nTraining baseline (raw text, SVM)...')
config.CLASSIFIER_TYPE = 'svm'
y_pred_baseline, _ = train_baseline_classifier(stage1_output['train'], stage1_output['test'], embedder)
results['baseline'] = compute_metrics(y_test, y_pred_baseline)
results['baseline']['per_article'] = per_article_f1(y_test, y_pred_baseline)
print(f'  {"baseline":<15} macro_f1={results["baseline"]["macro_f1"]:.4f}  micro_f1={results["baseline"]["micro_f1"]:.4f}')

config.CLASSIFIER_TYPE = original_clf_type

# Aggregate metrics table
metrics = ['macro_f1', 'micro_f1', 'macro_precision', 'macro_recall', 'micro_precision', 'micro_recall', 'hamming_loss']
print(f'\n{"="*66}')
print(f'{"Metric":<22} {"Baseline":>10} {"SVM":>10} {"DT":>10} {"EBM":>10}')
print(f'{"="*66}')
for m in metrics:
    row = f'{m:<22}'
    for key in ['baseline', 'svm', 'decision_tree', 'ebm']:
        row += f' {results[key][m]:>10.4f}'
    print(row)
print(f'{"="*66}')

# Per-article F1 table
print(f'\n{"="*66}')
print(f'{"Article":<16} {"Baseline":>10} {"SVM":>10} {"DT":>10} {"EBM":>10}')
print(f'{"="*66}')
for name in EVAL_ARTICLE_NAMES:
    row = f'{name:<16}'
    for key in ['baseline', 'svm', 'decision_tree', 'ebm']:
        pa = {r['article']: r['f1'] for r in results[key].get('per_article', [])}
        row += f' {pa.get(name, 0.0):>10.4f}'
    print(row)
print(f'{"="*66}')

Training classifiers:  33%|███▎      | 1/3 [00:34<01:09, 34.67s/it]

  svm             macro_f1=0.4949  micro_f1=0.5583


Training classifiers:  67%|██████▋   | 2/3 [02:20<01:16, 76.47s/it]

  decision_tree   macro_f1=0.3054  micro_f1=0.3210


## LegalBERT Classifier Comparison
Validates that Stage 1 premise extraction adds value: fine-tunes LegalBERT as a multi-label classifier on **extracted premises only** vs **full text**, then compares both against the explainable pipeline.

This is an additional experiment — our main pipeline remains the explainable SVM/DT/EBM approach.

In [16]:
from stage2_outcome_prediction.bert_classifier import train_bert_classifier, predict_bert
from evaluation.metrics import compute_metrics, per_article_f1, EVAL_ARTICLE_NAMES
import numpy as np

bert_results = {}

# --- Premises-only LegalBERT ---
print('Training LegalBERT on extracted premises...')
model_prem, tok_prem = train_bert_classifier(
    stage1_output['train'], stage1_output['val'],
    use_premises=True, epochs=3, batch_size=16)
y_pred_prem = predict_bert(model_prem, tok_prem, stage1_output['test'], use_premises=True)
y_test_bert = np.array([c['labels_binary'] for c in stage1_output['test']])
bert_results['premises_bert'] = compute_metrics(y_test_bert, y_pred_prem)
bert_results['premises_bert']['per_article'] = per_article_f1(y_test_bert, y_pred_prem)
print(f'  Premises LegalBERT  macro_f1={bert_results["premises_bert"]["macro_f1"]:.4f}  '
      f'micro_f1={bert_results["premises_bert"]["micro_f1"]:.4f}')

# --- Full-text LegalBERT ---
print('\nTraining LegalBERT on full text...')
model_full, tok_full = train_bert_classifier(
    stage1_output['train'], stage1_output['val'],
    use_premises=False, epochs=3, batch_size=16)
y_pred_full = predict_bert(model_full, tok_full, stage1_output['test'], use_premises=False)
bert_results['fulltext_bert'] = compute_metrics(y_test_bert, y_pred_full)
bert_results['fulltext_bert']['per_article'] = per_article_f1(y_test_bert, y_pred_full)
print(f'  Full-text LegalBERT macro_f1={bert_results["fulltext_bert"]["macro_f1"]:.4f}  '
      f'micro_f1={bert_results["fulltext_bert"]["micro_f1"]:.4f}')

# --- Comparison table ---
print(f'\n{"="*56}')
print(f'{"Metric":<22} {"Full-text":>15} {"Premises":>15}')
print(f'{"="*56}')
for m in ['macro_f1', 'micro_f1', 'macro_precision', 'macro_recall', 'hamming_loss']:
    f = bert_results['fulltext_bert'][m]
    p = bert_results['premises_bert'][m]
    print(f'{m:<22} {f:>15.4f} {p:>15.4f}')
print(f'{"="*56}')

print(f'\n{"="*56}')
print(f'{"Article":<16} {"Full-text":>15} {"Premises":>15}')
print(f'{"="*56}')
for name in EVAL_ARTICLE_NAMES:
    pa_f = {r['article']: r['f1'] for r in bert_results['fulltext_bert'].get('per_article', [])}
    pa_p = {r['article']: r['f1'] for r in bert_results['premises_bert'].get('per_article', [])}
    print(f'{name:<16} {pa_f.get(name, 0.0):>15.4f} {pa_p.get(name, 0.0):>15.4f}')
print(f'{"="*56}')

Training LegalBERT on extracted premises...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/bert-base-uncased-echr and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1/3: 100%|██████████| 563/563 [03:46<00:00,  2.48it/s]


  loss=0.2243  val_macro_f1=0.4369  val_micro_f1=0.6535


Epoch 2/3: 100%|██████████| 563/563 [03:46<00:00,  2.49it/s]


  loss=0.1397  val_macro_f1=0.5292  val_micro_f1=0.7047


...

  loss=0.2159  val_macro_f1=0.4322  val_micro_f1=0.6329


Epoch 2/3: 100%|██████████| 563/563 [04:27<00:00,  2.11it/s]


  loss=0.1390  val_macro_f1=0.5291  val_micro_f1=0.6954


Epoch 3/3: 100%|██████████| 563/563 [04:27<00:00,  2.10it/s]


  loss=0.1127  val_macro_f1=0.5965  val_micro_f1=0.6864
  Full-text LegalBERT macro_f1=0.5114  micro_f1=0.6170

Metric                       Full-text        Premises
macro_f1                        0.5114          0.5608
micro_f1                        0.6170          0.6396
macro_precision                 0.5484          0.6032
macro_recall                    0.4919          0.5381
hamming_loss                    0.0827          0.0783

Article                Full-text        Premises
Article 2                 0.8081          0.7925
Article 3                 0.7048          0.7990
Article 5                 0.6799          0.7508
Article 6                 0.6729          0.6579
Article 8                 0.5320          0.6299
Article 9                 0.0000          0.2000
Article 10                0.4961          0.4651
Article 11                0.6923          0.6761
Article 14                0.0000          0.1905
P1-1                      0.7295          0.7317
No Violation      

In [19]:
%matplotlib inline
from visualize import generate_all_plots

generate_all_plots(
    comparison=comparison,
    pa_f1_results=pa_f1_results,
    stage1_output=stage1_output,
    clf=clf,
    X_test=X_test,
)

Plots saved to: /home/sagemaker-user/NLPPW/outputs


## Save Outputs to S3
Uploads the fine-tuned checkpoint and all output artifacts back to S3 for safekeeping.

In [ ]:
import boto3
from pathlib import Path

s3     = boto3.client('s3')
OUTPUT = config.OUTPUT_DIR

def upload_dir_to_s3(local_dir: Path, s3_prefix: str):
    files = [f for f in local_dir.rglob('*') if f.is_file()]
    for f in files:
        key = f'{s3_prefix}/{f.relative_to(local_dir)}'
        s3.upload_file(str(f), S3_BUCKET, key)
    print(f'Uploaded {len(files)} files → s3://{S3_BUCKET}/{s3_prefix}/')

upload_dir_to_s3(OUTPUT, 'nlppw/outputs')
print('Done.')